<a href="https://colab.research.google.com/" target="_parent">Open in Google Colab</a>

# AI Agent Loop Demo

This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue toward a goal — all in a loop.

This notebook uses a small collection of recipe documents for the search tool.

---

## Setup - Gemini API key

How to get the key:

* Go to [Google AI Studio](https://aistudio.google.com/apikey)
* Sign in with your Google account
* Click **Create API key**
* Copy the key that appears

Save the key in Colab Secrets as `GOOGLE_API_KEY`. Do not share the key or upload it to GitHub.

## References
* [Recipe agent example](https://github.com/aditi-govindu/gemini-recipe-agent/blob/main/recipe_agent_gemini.ipynb)
* [Agent loop example](https://github.com/aditi-govindu/gemini-recipe-agent/blob/main/agent_loop_demo.ipynb)
* [Gemini function calling](https://ai.google.dev/gemini-api/docs/function-calling)

In [ ]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    key = userdata.get('GOOGLE_API_KEY')
    if not key:
        raise ValueError("API key not found in Colab Secrets")
    os.environ['GOOGLE_API_KEY'] = key.strip()
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key.strip()
    print('API key set.')

if not os.environ['GOOGLE_API_KEY']:
    raise ValueError("Please add a valid GOOGLE_API_KEY.")
del key

In [ ]:
# Quiet ensures your UI is not full of logs.
%pip install google-genai --quiet
print("Done.")

In [ ]:
# @title User input needed.

# Add at least four documents. Each document should have at least four sentences.
DOCUMENTS = {
    "vegetable_fried_rice": (
        "Vegetable fried rice is made with cooked rice, carrots, peas, onion and garlic. "
        "Day-old rice works well because it is less sticky than freshly cooked rice. "
        "First heat oil in a pan and cook the chopped vegetables until they become tender. "
        "Add garlic and then mix in the cooked rice with soy sauce, salt and pepper. "
        "Stir everything together for a few minutes and garnish with spring onion."
    ),
    "paneer_butter_masala": (
        "Paneer butter masala is prepared with paneer cubes and a tomato-based gravy. "
        "The gravy uses onions, tomatoes, butter, ginger, garlic and cashew nuts. "
        "Cook the onion and tomato mixture with the spices until it becomes soft. "
        "Blend the mixture and return it to the pan with butter and cream. "
        "Add paneer cubes and simmer for a few minutes before serving with naan or rice."
    ),
    "alfredo_pasta": (
        "Alfredo pasta is commonly made with fettuccine, butter, cream and Parmesan cheese. "
        "Cook the pasta in salted water until it is al dente. "
        "Melt butter in a pan and add cream over a low heat. "
        "Mix in grated Parmesan until the sauce becomes smooth. "
        "Add the cooked pasta, season with salt and pepper, and serve immediately."
    ),
    "fruit_custard": (
        "Fruit custard is made with milk, custard powder, sugar and fresh fruits. "
        "Mix custard powder with a little cold milk until there are no lumps. "
        "Heat the remaining milk with sugar and slowly add the custard mixture. "
        "Stir continuously until the mixture thickens and then allow it to cool. "
        "Add chopped fruits after cooling and keep the custard in the refrigerator before serving."
    )
}

In [ ]:
# @title Define a tool

def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check
print(search_documents("rice vegetables"))

In [ ]:
# @title Ask gemini to search for something
import time
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# The recipe-agent example uses Gemini Flash-Lite.
model_name = "gemini-3.5-flash-lite"

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find relevant information.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    system_instruction=(
        "You are a helpful cooking assistant. When you need information about "
        "ingredients or cooking steps, use the search_documents tool. "
        "Answer using the documents. If the documents do not contain the answer, "
        "say so clearly."
    ),
    thinking_config=types.ThinkingConfig(thinking_level="minimal"),
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

print("Model ready:", model_name)

In [ ]:
import time
import re
from google.genai import types


class DailyQuotaExceeded(Exception):
    pass


def send_with_retry(contents, max_retries=2):
    for attempt in range(max_retries):
        try:
            start_time = time.perf_counter()
            response = client.models.generate_content(
                model=model_name,
                contents=contents,
                config=config
            )
            print(f"Gemini request completed in {time.perf_counter() - start_time:.1f} seconds")
            return response
        except Exception as e:
            error = str(e)
            error_lower = error.lower()

            # A daily quota error will not be fixed by waiting a few seconds.
            daily_quota = (
                "generaterequestsperday" in error_lower or
                "generatecontentinputtokenspermodelperday" in error_lower or
                "requests per day" in error_lower or
                "daily quota" in error_lower
            )
            if daily_quota:
                raise DailyQuotaExceeded(
                    "Gemini's daily free-tier request limit has been reached. "
                    "No short retry will help. Try again after the quota resets."
                ) from e

            # Retry a temporary rate limit or server error only once.
            temporary_error = ("429" in error or "503" in error or "500" in error)
            if temporary_error and attempt < max_retries - 1:
                match = re.search(r"retry in ([0-9.]+)s", error_lower)
                if "429" in error and match:
                    wait = min(float(match.group(1)), 15)
                else:
                    wait = 3
                print(f"Request busy, waiting {wait:g} seconds...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 3) -> str:
    start_time = time.perf_counter()
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn so the function call is kept in the conversation.
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            print(f"Total time for this question: {time.perf_counter() - start_time:.1f} seconds")
            return response.text

        # Run the requested tool and send the result back.
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            if call.name == "search_documents":
                result = search_documents(**args)
            else:
                result = "Unknown tool."
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )

        # The starter example sends the function result in a user message.
        contents.append(types.Content(role="user", parts=result_parts))

    print(f"Total time for this question: {time.perf_counter() - start_time:.1f} seconds")
    return "(Stopped: hit max_steps)"


try:
    _ = run_agent("Which ingredients are used in vegetable fried rice?")
except DailyQuotaExceeded as e:
    print(e)

In [ ]:
# @title Ask follow up questions

# Enter questions based on the documents for the agent to answer.
questions = [
    "Which vegetables are used in vegetable fried rice?",
    "How is paneer butter masala prepared?",
    "What ingredients are used in Alfredo pasta?",
    # This question is not answered by the documents.
    "How many customers eat at the restaurant every day?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # pause between questions to stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    try:
        run_agent(q)
    except DailyQuotaExceeded as e:
        print(e)
        print("The remaining questions were not run.")
        break

In [ ]:
# This cell will check that you have added enough documents and questions.
assert(len(DOCUMENTS) >= 4)
assert(len(questions) >= 4)
assert("Add a question here." not in questions)
print("Checks passed.")

# Next steps:
* Save a copy of this notebook to your Google Drive.
* Download the notebook and upload it to your public GitHub repo.
* Do not upload your API key to GitHub.